# Ball Possession Pipeline

This notebook runs the whole pipeline and shows the results. Every step calls the `possession` package, so the notebook and the command line give the same output.

Run the cells from top to bottom. Clear all outputs before committing. The match data may not be published, and outputs can contain parts of it.

## Settings

The notebook defines no model setting of its own. The values below are read from the package, where the command line reads them too, and the table under the cell shows them.

| Setting | Meaning |
|---|---|
| `TRAIN` | `False` uses the weights in `models/` and takes about 22 minutes. `True` trains and adapts first, which takes about 4 to 5 hours on an RTX 4060 and replaces those weights |
| `EPOCHS`, `STEPS` | Training length: epochs per network and batches per epoch. Lower both for a test run |
| `SEEDS` | One network is trained per seed, and their outputs are averaged |
| `NAME` | File name of the weights and the decoder setting in `models/` |
| `COPIES` | How many degraded copies of an open match the chart in section 4 draws |
| `BIN_SECONDS` | Bar width of the histogram in section 8 |
| `COLOURS` | One plot colour per label, in the order of `LABELS` |

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch

from possession import adapt, predict, train
from possession.broadcast import half_lags, target_match
from possession.degrade import degrade
from possession.dfl import MATCHES, load_dfl
from possession.features import FEATURE_NAMES, match_features
from possession.match import AWAY, DEAD, HOME, LABELS
from possession.metrics import spells
from possession.paths import FPS, PUBLIC_GT, RESULTS_DIR, ROOT, TEAM
from possession.tracking import load_tracking

TRAIN = False
EPOCHS, STEPS, SEEDS, NAME = train.EPOCHS, train.STEPS, train.SEEDS, train.NAME
COPIES = 3
BIN_SECONDS = 2
COLOURS = dict(zip(LABELS.tolist(), ("tab:blue", "tab:red", "tab:gray"), strict=True))

pl.DataFrame(
    {
        "Setting": ["TRAIN", "EPOCHS", "STEPS", "SEEDS", "NAME", "Held-Out Open Matches", "Prediction Device"],
        "Value": [str(value) for value in (TRAIN, EPOCHS, STEPS, SEEDS, NAME, train.VALID_IDS, predict.DEVICE)],
    }
)

## Hardware

Training uses the GPU when PyTorch can see one, and falls back to the CPU otherwise. Prediction always runs on the CPU, because GPU inference gave a different file and the submitted file must be reproducible.

In [ ]:
gpu = torch.cuda.is_available()
memory = torch.cuda.get_device_properties(0).total_memory / 1e9 if gpu else 0.0
pl.DataFrame(
    {
        "Item": [
            "PyTorch Version",
            "CUDA Available",
            "GPU",
            "GPU Memory in GB",
            "Training Device",
            "Prediction Device",
        ],
        "Value": [
            torch.__version__,
            str(gpu),
            torch.cuda.get_device_name(0) if gpu else "None",
            f"{memory:.1f}" if gpu else "None",
            "cuda" if gpu else "cpu",
            predict.DEVICE,
        ],
    }
)

## 1. Provided Tracking

The organisers provide two recordings of the match, a tactical wide shot and the television broadcast, each with automatic tracking. The table shows how often each tracker sees the ball and how many players it follows per frame.

In [ ]:
rows = []
for source in ("tactical", "broadcast"):
    tracking = load_tracking(source)
    seen = ~np.isnan(tracking.xy[:, :, 0])
    rows.append(
        {
            "Source": source,
            "Frames": tracking.n_frames,
            "Track Slots": len(tracking.team),
            "Ball Seen": round(float((~np.isnan(tracking.ball[:, 0])).mean()), 3),
            "Players per Frame": round(float(seen.sum(axis=1).mean()), 1),
        }
    )
pl.DataFrame(rows)

## 2. Broadcast Alignment

The broadcast runs on its own timeline. The lag of each half is found by cross correlating the two ball tracks. After alignment the broadcast gives a second ball track and a cutaway flag, which is set when the broadcast shows a replay or a close-up.

In [ ]:
match = target_match()
graded = np.zeros(match.n_frames, dtype=bool)
for start, stop in match.blocks:
    graded[start:stop] = True

first = ~np.isnan(match.ball[graded, 0])
second = ~np.isnan(match.second[graded, 0])
pl.DataFrame(
    {
        "Measure": ["Lag of Each Half in Frames", "Tactical Ball Seen", "Either Ball Seen", "Cutaway Share"],
        "Value": [
            str(half_lags()),
            f"{first.mean():.3f}",
            f"{(first | second).mean():.3f}",
            f"{match.cutaway[graded].mean():.3f}",
        ],
    }
)

## 3. Open Training Matches

Seven matches of the German Bundesliga and 2. Bundesliga from the DFL open dataset (Bassek et al. 2025, CC-BY 4.0). They are downloaded on first use. The matches marked as held out are not trained on in step 1 of training. They are used to tune the decoder.

In [ ]:
rows = []
for match_id in MATCHES:
    open_match, _ = load_dfl(match_id)
    share = np.bincount(open_match.label, minlength=len(LABELS)) / open_match.n_frames
    rows.append(
        {
            "Match": match_id,
            "Held Out": match_id in train.VALID_IDS,
            "Frames": open_match.n_frames,
            "Home": round(float(share[HOME]), 3),
            "Away": round(float(share[AWAY]), 3),
            "Dead Ball": round(float(share[DEAD]), 3),
            "Possessions": len(spells(open_match.label, open_match.blocks)),
        }
    )
pl.DataFrame(rows)

## 4. Degraded Open Data Against the Target

The open tracking has none of the gaps of automatic tracking, so it is degraded before training. The chart compares the mean of some features on the target match with `COPIES` degraded copies of one open match. The copies should surround the target. The open match is the first one that is not held out.

The features are picked by name from `FEATURE_NAMES` in `possession/features.py`, which lists all 50 in column order. The ones shown are those the degradation changes most: ball visibility, ball motion, the second camera, cutaways, and lost player tracks.

In [ ]:
shown = [
    "Ball Seen",
    "Ball Gap Length",
    "Ball Speed",
    "Ball Acceleration",
    "Both Balls Seen",
    "Ball Disagreement",
    "Cutaway",
    "Home Nearest Player",
    "Home Players Seen",
    "Away Nearest Player",
    "Away Players Seen",
]
columns = [FEATURE_NAMES.index(name) for name in shown]
target_mean = match_features(match)[graded][:, columns].mean(axis=0)

example = next(match_id for match_id in MATCHES if match_id not in train.VALID_IDS)
open_match, height = load_dfl(example)
copies = [match_features(degrade(open_match, height, seed))[:, columns].mean(axis=0) for seed in range(COPIES)]

# One bar per group member
groups = [("Target Match", target_mean)] + [
    (f"Degraded Copy {number + 1}", values) for number, values in enumerate(copies)
]
width = 0.8 / len(groups)
position = np.arange(len(columns))
figure, axis = plt.subplots(figsize=(11, 4))
for number, (name, values) in enumerate(groups):
    offset = (number - (len(groups) - 1) / 2) * width
    axis.bar(position + offset, values, width, label=name, color="black" if number == 0 else None)
axis.set_xticks(position, shown, rotation=30, ha="right")
axis.set_ylabel("Mean Feature Value")
axis.set_title(f"Feature Means, Target Against Degraded Copies of {example}")
axis.legend()
plt.show()

## 5. Training

This cell only trains when `TRAIN` is `True`. Otherwise the weights and the decoder setting in `models/` are used.

Step 1 fits one network per seed on five matches. The decoder is then tuned on two validation sets that are never trained on: the two held-out open matches, and 20 real matches with automatic tracking from the SkillCorner open data (MIT licence, 1.8 GB, downloaded on first use). The two sets label possession differently, so the setting with the smallest loss on its worse set is kept. Step 2 refits every network on all seven matches and saves the weights to `models/`. Each epoch trains on fresh degraded copies of every match, built by worker processes.

After training, each network is adapted to the provided match. The averaged networks label the match, the frames where they are confident are kept, and each network trains 800 more steps on them. No ground truth and no hand label is involved. The adapted weights are saved next to the trained ones and are only used for the team decision in section 6.

Do not edit or delete the training cell while it runs. The decoder tuning in the middle takes about 11 minutes and prints one line per switch cost.

In [ ]:
if TRAIN:
    scores = train.run(epochs=EPOCHS, steps=STEPS, seeds=SEEDS, name=NAME)
    print(f"spellF1 open held-out {scores[0]:.4f}, real matches {scores[1]:.4f}")
    adapt.run(NAME)
else:
    print((ROOT / "models" / f"{NAME}_decoder.json").read_text())

## 6. Submission File

Prediction runs on CPU, which gave the same file on every run we tried. It takes about 20 minutes, because it averages eight mirrored views of the match for each network, once for the trained networks and once for the adapted ones. The stoppage probability comes from the trained networks, and the adapted networks only split the rest between the two teams. The public window score is only a check that the pipeline works. It is not used for tuning.

In [ ]:
labels, prob = predict.run(models=(NAME,), decoder=f"{NAME}_decoder", adapted=predict.ADAPTED)

In [ ]:
submission = RESULTS_DIR / f"{TEAM}.csv"
for command in (
    ["evaluation/validate_submission.py", str(submission)],
    ["evaluation/run_evaluation.py", "--no-detail"],
):
    result = subprocess.run([sys.executable, *command], cwd=ROOT, capture_output=True, text=True, check=False)
    print(result.stdout[-2500:], result.stderr[-500:])

## 7. Public Window

The organisers published the ground truth for the first five minutes of play only. The rest of the match is graded in private. This plot can therefore only compare labels inside that window, although the submission file labels the whole match.

The top panel shows the ground truth above our labels. The middle panel shows the probability of each label. The bottom panel shows when either camera sees the ball and when the broadcast cuts away.

In [ ]:
truth = pl.read_csv(PUBLIC_GT)["label"].to_numpy()
window = np.flatnonzero(np.isin(truth, LABELS))
seconds = (window - window[0]) / FPS
ours = LABELS[labels[window]]

figure, axes = plt.subplots(3, 1, figsize=(13, 6), sharex=True, height_ratios=[1, 3, 1])
rows = {"Ground Truth": truth[window], "Prediction": ours}
for row, values in enumerate(rows.values()):
    for label, colour in COLOURS.items():
        axes[0].fill_between(seconds, row, row + 0.9, where=values == label, color=colour, step="post")
axes[0].set_yticks(np.arange(len(rows)) + 0.45, list(rows))
axes[0].invert_yaxis()
axes[0].set_title("Labels in the Public Window")

for code, (label, colour) in enumerate(COLOURS.items()):
    axes[1].plot(seconds, prob[window, code], color=colour, label=label, linewidth=0.8)
axes[1].set_ylabel("Probability")
axes[1].legend(loc="upper right", ncol=len(COLOURS))

seen = ~np.isnan(match.ball[window, 0]) | ~np.isnan(match.second[window, 0])
flags = {"Cutaway": (match.cutaway[window], "tab:orange"), "Ball Seen": (seen, "tab:green")}
for row, (flag, colour) in enumerate(flags.values()):
    axes[2].fill_between(seconds, row, row + 0.9, where=flag, color=colour, step="post")
axes[2].set_yticks(np.arange(len(flags)) + 0.45, list(flags))
axes[2].set_xlabel("Seconds From Kickoff")
plt.tight_layout()
plt.show()

## 8. Predicted Possessions

The ranking metric counts whole possessions, so their number and length matter more than frame accuracy. This section covers the whole match, not only the public window. The histogram uses bars of `BIN_SECONDS` and stops at the longest possession.

In [ ]:
found = spells(labels, match.blocks)
lengths = np.array([stop - start for start, stop, _ in found]) / FPS
in_blocks = labels[graded]
alive = in_blocks != DEAD
summary = pl.DataFrame(
    {
        "Measure": [
            "Possessions",
            "Mean Length in Seconds",
            "Median Length in Seconds",
            "Dead Ball Share",
            "Home Share of Live Play",
        ],
        "Value": [
            f"{len(found)}",
            f"{lengths.mean():.1f}",
            f"{np.median(lengths):.1f}",
            f"{(~alive).mean():.3f}",
            f"{(in_blocks[alive] == HOME).mean():.3f}",
        ],
    }
)

figure, axis = plt.subplots(figsize=(9, 3.5))
axis.hist(lengths, bins=np.arange(0, lengths.max() + BIN_SECONDS, BIN_SECONDS), color=COLOURS[LABELS[HOME]])
axis.set_xlabel("Possession Length in Seconds")
axis.set_ylabel("Number of Possessions")
axis.set_title("Length of Predicted Possessions")
plt.show()
summary